# 07 · TMA prefetch into L2

Notebook 02 prefetched by issuing TMA copies earlier. This is a different lever: `cute.prefetch()` pulls tile data into **L2** ahead of the TMA copy that will consume it, so the copy itself hits cache instead of DRAM.

A prefetch that lands too late is worthless. The trace tells you whether the ordering you intended is the ordering you got.

## 1. The question

`fp16_gemm_5` prefetches in two places:

- **Initial** — issue prefetches for the first `prefetch_dist` K-tiles before the mainloop starts
- **Rolling** — inside each K-tile iteration, prefetch `prefetch_dist` tiles ahead

**The question:** is the prefetch positioned early enough to actually hide DRAM latency — does it land *before* the TMA issue that needs it?

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        610.57.04
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


## 2. The instrumentation

Three annotation points, which together answer an ordering question rather than a duration question:

```python
iket.range_push("initial_prefetch")   # before the mainloop
for pf_k in range(prefetch_dist):
    cute.prefetch(tma_a.atom, future_tile)
iket.range_pop()

iket.range_push("rolling_prefetch")   # inside each K-tile
cute.prefetch(tma_a.atom, k_tile + prefetch_dist)
iket.range_pop()
```

Because IKET timestamps are wall-clock nanoseconds on a common timebase, you can compare *when* `initial_prefetch` ended against *when* the first `tma_issue` happened — a question about causality, not cost.

In [2]:
nbhelp.show_iket_calls(
    nbhelp.kernel("instrumented", "fp16_gemm_5_iket_detailed_marks.py"),
    limit=12,
)

fp16_gemm_5_iket_detailed_marks.py: 51 IKET call sites (showing first 12)
------------------------------------------------------------------------
>  131      cute.experimental.iket.range_push("prologue")
       ...
>  382      cute.experimental.iket.range_pop()
       ...
>  400              cute.experimental.iket.range_push("tma_tile")
       ...
>  421              cute.experimental.iket.range_push("initial_prefetch")
       ...
>  425                  cute.experimental.iket.mark("prefetch_issue")  # async op -> mark
       ...
>  428              cute.experimental.iket.range_pop()
       ...
>  435                  cute.experimental.iket.range_push("prod_acquire")
       ...
>  437                  cute.experimental.iket.range_pop()
       ...
>  440                  cute.experimental.iket.mark("tma_issue")  # async op -> mark
       ...
>  460                      cute.experimental.iket.range_push("rolling_prefetch")
>  461                      cute.experimental.iket.mark("prefetc

## 3. Run it

In [3]:
OUT = "out/nb07_gemm5"
cmd = (
    f"run-iket --output-dir {OUT} --clobber profile --postprocess all -- "
    f"python {nbhelp.kernel("instrumented", "fp16_gemm_5_iket_detailed_marks.py")} --mnk 4096,4096,1024"
)
check_env.run_or_show(cmd, env, est="about 2 minutes")

$ run-iket --output-dir out/nb07_gemm5 --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_5_iket_detailed_marks.py --mnk 4096,4096,1024

[run-iket] ============= Dry run application first to collect some info... =============


Running Blackwell fp16 GEMM example 5 (with TMA prefetch):
  mnk:       [4096, 4096, 1024]
  tolerance: 0.1

/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_5_iket_detailed_marks.py:750: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
PASS
[run-iket] Auto-computed context buffer size: 0.10 GB (104,721,612 bytes)
[run-iket] ============= Run application for real profiling... =============


Running Blackwell fp16 GEMM example 5 (with TMA prefetch):
  mnk:       [4096, 4096, 1024]
  tolerance: 0.1

/path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_5_iket_detailed_marks.py:750: DeprecationWarning: Migrated to examples/CuTeDSL/helpers/dynamic_persistent_tile_scheduler.py (BSD-3). The wheel copy will be removed in a future release.
  tile_sched_params = utils.ClcDynamicPersistentTileSchedulerParams(
PASS


[run-iket] Dumped perfetto trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb07_gemm5/iket_pid_0x3920f2.pftrace
[run-iket] Dumped json trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb07_gemm5/iket_pid_0x3920f2.trace.json


[run-iket] Dumped compressed trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb07_gemm5/iket_pid_0x3920f2.pftrace.gz
[run-iket] Dumped perfetto HTML viewer to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb07_gemm5/iket_pid_0x3920f2.html
[run-iket] To view: cd /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb07_gemm5 && python3 -m http.server 8080 , then open http://localhost:8080/iket_pid_0x3920f2.html
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb07_gemm5/tracker
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb07_gemm5/gen-config
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb07_gemm5/iket


0

## 4. The trace

In [4]:
nbhelp.figure("nb07/gemm5_initial_rolling_prefetch.png", "initial_prefetch primes L2 before the mainloop; rolling_prefetch runs ahead inside it.")

In [5]:
try:
    trace = ts.load(ts.find_trace("out/nb07_gemm5"))
    ts.print_summary(ts.summarize(trace), top=12)
except FileNotFoundError:
    print("No local trace here — the reference numbers below were measured on a NVIDIA B300.")

kernel_cutlass_kernel_TiledMMA_ThrLayoutVMNK21111000_PermutationMNK____MMAAtom_ThrID21_ShapeMNK2566416_TVLayoutA2128161281256_TVLayoutB2321632164_TVLayoutC2128641281256_cutlasscutenvgpucp_0   grid 32x64x1  block 224x1x1  warps traced 924
warp-lifetime span: 47.2 us

range                      count      median         p90       total
--------------------------------------------------------------------
epi_tile                    8192      2.784u      3.040u   23113.54u
acc_wait                    8192      2.176u      2.400u   18281.44u
tma_tile                    2048      2.752u      2.912u    5661.28u
mma_clc_wait                2048      0.064u      2.880u    2773.60u
mma_tile                    1024      2.688u      2.848u    2773.54u
prod_acquire               32768      0.064u      0.064u    1962.53u
cons_wait                  16384      0.064u      0.256u    1692.35u
clc_prod_acquire             512      2.528u      2.784u    1256.38u
initial_prefetch            2048      0.384

## 5. The finding

Measured on a NVIDIA B300 at `mnk=4096,4096,1024` — 924 warps, 47.26 µs span:

| Range | Median | Count | Reading |
|---|---|---|---|
| `initial_prefetch` | 0.384 µs | 2,048 | Primes L2 before the first TMA issue |
| `rolling_prefetch` | 0.032 µs | 12,288 | One quantum — essentially free per tile |
| `cons_wait` | **0.064 µs** | 16,384 | The MMA warp is not starved |
| `tma_tile` | 2.752 µs | 2,048 | The TMA work itself |


**The prefetch lands where it was meant to.** `initial_prefetch` completes before the first `tma_issue`, and `cons_wait` stays at 0.064 µs across 16,384 iterations — the same low value notebook 02 achieved with a well-tuned pipeline. The consumer is fed.

**`rolling_prefetch` costs one timestamp quantum.** At 0.032 µs it is below the resolution of this instrumentation: issuing a prefetch is nearly free, which is what makes prefetching ahead a reasonable default rather than a trade-off.

### An honest limitation

**This is a capability demonstration, not a controlled A/B.** `fp16_gemm_5` also changes tile shape and uses a 2×2 cluster relative to earlier kernels, so its 47.26 µs cannot be attributed to prefetch alone. What this notebook legitimately shows is that prefetch *placement is observable* — you can confirm the ordering you intended.

Turning that into a causal claim would need a variant that changes `prefetch_dist` and nothing else, in the style of notebook 02. That experiment is not in this series, and saying so is more useful than a number that would not survive scrutiny.

**Next:** [08 · PDL — when the profiler proves you wrong](08_pdl.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*